# Name Entity Recognition using Deep Learning

* Upload the lab_resources and NERC_nn files to you Drive Account:
  * Lab_resource: https://www.cs.upc.edu/~turmo/mud/lab/lab_resources.zip
  * NERC_nn code: https://www.cs.upc.edu/~turmo/mud/lab/06-NERC-nn.zip
  
* Before running the code, ensure that your Google Colab is set to use GPU:
  * Edit → Notebook Settings
* Mount your Drive disk unit:
  * Left-side menu → Files → Mount drive (the icon that looks like a folder with the Drive logo).


In [1]:
! pip install tensorflow
! pip install keras

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Define the paths to the data and utils in your Drive unit:

In [3]:
utilsdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 2/06-NERC-nn/06-NERC-nn'
evaluatordir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 2/lab_resources/DDI/util'
traindir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 2/lab_resources/DDI/data/train'
validationdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 2/lab_resources/DDI/data/devel'
testdir='/content/drive/MyDrive/Data Science/MUD/LAB MUD/Task 2/lab_resources/DDI/data/test'
modelname ='model.keras'
outfile ='out.txt'

In [4]:
!pip install tensorflow-addons

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons


In [4]:
import os
import io
import random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

import sys
sys.path.insert(1,utilsdir) # Path to the utils folder on your Google Drive disk
sys.path.insert(1,evaluatordir) # Path to the evaluator folder on your Google Drive disk

In [5]:
from contextlib import redirect_stdout

from tensorflow.keras import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, GRU, Embedding, Dense, TimeDistributed, Dropout, Bidirectional, concatenate, Softmax
from codemaps import *

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
# Download GloVe vectors
!wget https://nlp.stanford.edu/data/glove.2024.wikigiga.200d.zip
!unzip -q glove.2024.wikigiga.200d.zip

!ls

--2026-03-24 14:53:48--  https://nlp.stanford.edu/data/glove.2024.wikigiga.200d.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.2024.wikigiga.200d.zip [following]
--2026-03-24 14:53:49--  https://downloads.cs.stanford.edu/nlp/data/glove.2024.wikigiga.200d.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1149444870 (1.1G) [application/zip]
Saving to: ‘glove.2024.wikigiga.200d.zip’

glove.2024.wikigiga 100%[===================>]   1.07G  5.02MB/s    in 3m 36s  

2026-03-24 14:57:24 (5.09 MB/s) - ‘glove.2024.wikigiga.200d.zip’ saved [1149444870/1149444870]

drive
glove.2024.wikigiga.

In [7]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.2 MB/s eta 0:00:00


In [8]:
from gensim.models import KeyedVectors

GLOVE_PATH = "wiki_giga_2024_200_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt"

glove_vectors = KeyedVectors.load_word2vec_format(GLOVE_PATH, binary=False, no_header=True)
print("Loaded GloVe vectors:", len(glove_vectors.key_to_index))
print("Vector size:", glove_vectors.vector_size)

Loaded GloVe vectors: 1291147
Vector size: 200


In [ ]:
# directory with files to process
# load train and validation data
traindata = Dataset(traindir)
valdata = Dataset(validationdir)

# create indexes from training data
max_len = 200 # baseline = 150
suf_len = 5 # baseline = 5
codes = Codemaps(traindata, max_len, suf_len)

# encode datasets
[Xt, Xts, Xtlw, Xtp, Xtf] = codes.encode_words(traindata)
Yt = codes.encode_labels(traindata)

[Xv, Xvs, Xvlw, Xvp, Xvf] = codes.encode_words(valdata)
Yv = codes.encode_labels(valdata)

n_tags = codes.get_n_labels()
max_len = codes.maxlen

In [ ]:
import numpy as np

def build_pretrained_matrix(word_index, pretrained_vectors, emb_dim):
    vocab_size = len(word_index)
    matrix = np.random.normal(scale=0.05, size=(vocab_size, emb_dim)).astype(np.float32)

    # PAD a zeros
    if "PAD" in word_index:
        matrix[word_index["PAD"]] = np.zeros(emb_dim, dtype=np.float32)

    found = 0
    for word, idx in word_index.items():
        if word in ["PAD", "UNK"]:
            continue
        if word in pretrained_vectors:
            matrix[idx] = pretrained_vectors[word]
            found += 1

    print(f"Found {found} pretrained vectors out of {vocab_size} entries.")
    return matrix

GLOVE_DIM = 200
pretrained_lc_matrix = build_pretrained_matrix(codes.lc_word_index, glove_vectors, GLOVE_DIM)

Found 7605 pretrained vectors out of 8339 entries.


In [ ]:
# Architecture

def build_network(codes, pretrained_lc_matrix=None, lc_trainable=False):

   # sizes
   n_words = codes.get_n_words()
   n_lc_words = codes.get_n_lc_words()
   n_sufs = codes.get_n_sufs()
   n_prefs = codes.get_n_prefs()
   n_labels = codes.get_n_labels()
   max_len = codes.maxlen

   ######################################
   inptW = Input(shape=(max_len,))
   inptS = Input(shape=(max_len,))
   inptLW = Input(shape=(max_len,))
   inptP = Input(shape=(max_len,))
   inptF = Input(shape=(max_len, 8))

   model1 = Embedding(input_dim=n_words, output_dim=200,
                      input_length=max_len, mask_zero=False)(inptW)  # word embeddings, baseline = 150
   model2 = Embedding(input_dim=n_sufs, output_dim=100,
                      input_length=max_len, mask_zero=False)(inptS)  # suf embeddings, baseline = 50

   if pretrained_lc_matrix is not None:
        model3 = Embedding(
            input_dim=n_lc_words,
            output_dim=pretrained_lc_matrix.shape[1],
            weights=[pretrained_lc_matrix],
            trainable=lc_trainable,
            input_length=max_len,
            mask_zero=False
        )(inptLW) # pretrained embeddings
   else:
        model3 = Embedding(
            input_dim=n_lc_words,
            output_dim=100,
            input_length=max_len,
            mask_zero=False
        )(inptLW)  # lowercase embeddings

   model4 = Embedding(input_dim=n_prefs, output_dim=100,
                       input_length=max_len, mask_zero=False)(inptP)   # prefix embeddings

   model1 = Dropout(0.1)(model1)
   model2 = Dropout(0.1)(model2)
   model3 = Dropout(0.1)(model3)
   model4 = Dropout(0.1)(model4)

   model = concatenate([model1,model2,model3,model4,inptF])

   #y = Bidirectional(LSTM(units=200, return_sequences=True))(model)  # biLSTM, baseline = 200
   #out = TimeDistributed(Dense(n_labels, activation=Softmax()))(y)

   y = Bidirectional(LSTM(units=200, return_sequences=True))(model)
   y = TimeDistributed(Dense(128, activation="relu"))(y)
   #y = Dropout(0.2)(y)
   out = TimeDistributed(Dense(n_labels, activation=Softmax()))(y)

   #y = Bidirectional(LSTM(units=200, return_sequences=True))(model)
   #y = Bidirectional(LSTM(units=100, return_sequences=True))(y)
   #out = TimeDistributed(Dense(n_labels, activation=Softmax()))(y)

   return Model(inputs=[inptW,inptS,inptLW,inptP,inptF], outputs=out)


In [ ]:
#model = build_network(codes, pretrained_lc_matrix=pretrained_lc_matrix, lc_trainable=False) # frozen pretrained embeddings
model = build_network(codes, pretrained_lc_matrix=pretrained_lc_matrix, lc_trainable=True) # fine-tuned pretrained embeddings

model.compile(optimizer='adam' ,metrics=["accuracy"], loss="sparse_categorical_crossentropy") # baseline optimizer = 'adam'
model.build([
    (None, max_len),
    (None, max_len),
    (None, max_len),
    (None, max_len),
    (None, max_len, 8)]) # add new inputs

with redirect_stdout(sys.stderr) :
   model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 200, 200)  │  1,935,200 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 200, 100)  │    495,700 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 200, 200)  │  1,667,800 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 200, 100)  │    517,500 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 200, 200)  │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 200, 100)  │          0 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 200, 200)  │          0 │ embedding_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 200, 100)  │          0 │ embedding_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, 200, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 200, 608)  │          0 │ dropout_4[0][0],  │
│ (Concatenate)       │                   │            │ dropout_5[0][0],  │
│                     │                   │            │ dropout_6[0][0],  │
│                     │                   │            │ dropout_7[0][0],  │
│                     │                   │            │ input_layer_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 200, 400)  │  1,294,400 │ concatenate_1[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 200, 128)  │     51,328 │ bidirectional_1[… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 200, 10)   │      1,290 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                 

 Total params: 5,963,218 (22.75 MB)

 Trainable params: 5,963,218 (22.75 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
## --------- MAIN PROGRAM -----------
## --
## -- Usage:  train.py ../data/Train ../data/Devel  modelname
## --

# train model
with redirect_stdout(sys.stderr) :
   model.fit([Xt, Xts, Xtlw, Xtp, Xtf], Yt,
          batch_size=32,
          epochs=10,
          validation_data=([Xv, Xvs, Xvlw, Xvp, Xvf], Yv),
          verbose=1)

# save model and indexs
model.save(modelname)
codes.save(modelname)
#save_model_and_indexs(model, idx, modelname)

Epoch 1/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 42s 137ms/step - accuracy: 0.9856 - loss: 0.0602 - val_accuracy: 0.9950 - val_loss: 0.0199
Epoch 2/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 17s 101ms/step - accuracy: 0.9977 - loss: 0.0085 - val_accuracy: 0.9962 - val_loss: 0.0166
Epoch 3/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 17s 101ms/step - accuracy: 0.9988 - loss: 0.0044 - val_accuracy: 0.9960 - val_loss: 0.0210
Epoch 4/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 19s 113ms/step - accuracy: 0.9992 - loss: 0.0029 - val_accuracy: 0.9963 - val_loss: 0.0194
Epoch 5/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 17s 101ms/step - accuracy: 0.9995 - loss: 0.0019 - val_accuracy: 0.9963 - val_loss: 0.0208
Epoch 6/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 17s 101ms/step - accuracy: 0.9997 - loss: 0.0013 - val_accuracy: 0.9964 - val_loss: 0.0194
Epoch 7/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 17s 102ms/step - accuracy: 0.9997 - loss: 9.4858e-04 - val_accuracy: 0.9965 - val_loss: 0.0180
Epoch 8/10
170/170 ━━━━━━━━━━━━━━━━━━━━ 18s 103ms/step - accuracy: 0.9998 - los

# Predict

In [ ]:
#import sys
import evaluator

In [ ]:
def output_entities(data, preds, outfile) :

   outf = open(outfile, 'w')
   for sid,tags in zip(data.sentence_ids(),preds) :
      inside = False
      for k in range(0,min(len(data.get_sentence(sid)),codes.maxlen)) :
         y = tags[k]
         token = data.get_sentence(sid)[k]

         if (y[0]=="B") :
             entity_form = token['form']
             entity_start = token['start']
             entity_end = token['end']
             entity_type = y[2:]
             inside = True
         elif (y[0]=="I" and inside) :
             entity_form += " "+token['form']
             entity_end = token['end']
         elif (y[0]=="O" and inside) :
             print(sid, str(entity_start)+"-"+str(entity_end), entity_form, entity_type, sep="|", file=outf)
             inside = False

      if inside : print(sid, str(entity_start)+"-"+str(entity_end), entity_form, entity_type, sep="|", file=outf)

   outf.close()

In [ ]:
## --------- Evaluator -----------
def evaluation(datadir,outfile) :
   evaluator.evaluate("NER", datadir, outfile)


In [ ]:
## --------- MAIN PROGRAM -----------
## --
## -- Usage:  baseline-NER.py target-dir
## --
## -- Extracts Drug NE from all XML files in target-dir
## --

# Validation data
datadir = validationdir

testdata = Dataset(datadir)
[X, Xs, Xlw, Xp, Xf] = codes.encode_words(testdata)
Y = model.predict([X, Xs, Xlw, Xp, Xf])
Y = [[codes.idx2label(np.argmax(w)) for w in s] for s in Y]

# extract entities
output_entities(testdata, Y, outfile)

# evaluate
evaluation(datadir,outfile)

45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step
                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
brand             134	   6	 240	 140	 374	95.7%	35.8%	52.1%
drug             1402	  59	 504	1461	1906	96.0%	73.6%	83.3%
drug_n              9	   9	  36	  18	  45	50.0%	20.0%	28.6%
group             533	  91	 154	 624	 687	85.4%	77.6%	81.3%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	81.8%	51.7%	61.3%
------------------------------------------------------------------------------
m.avg            2078	 165	 934	2243	3012	92.6%	69.0%	79.1%
m.avg(no class)  2115	 128	 897	2243	3012	94.3%	70.2%	80.5%


In [ ]:
# Test data
datadir = testdir

testdata = Dataset(datadir)
[X, Xs, Xlw, Xp, Xf] = codes.encode_words(testdata)
Y = model.predict([X, Xs, Xlw, Xp, Xf])
Y = [[codes.idx2label(np.argmax(w)) for w in s] for s in Y]

# extract entities
output_entities(testdata, Y, outfile)

# evaluate
evaluation(datadir,outfile)

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
                   tp	  fp	  fn	#pred	#exp	P	R	F1
------------------------------------------------------------------------------
brand             126	  16	 148	 142	 274	88.7%	46.0%	60.6%
drug             1616	  50	 511	1666	2127	97.0%	76.0%	85.2%
drug_n              0	  20	  72	  20	  72	0.0%	0.0%	0.0%
group             553	 121	 140	 674	 693	82.0%	79.8%	80.9%
------------------------------------------------------------------------------
M.avg            -	-	-	-	-	66.9%	50.4%	56.7%
------------------------------------------------------------------------------
m.avg            2295	 207	 871	2502	3166	91.7%	72.5%	81.0%
m.avg(no class)  2353	 149	 813	2502	3166	94.0%	74.3%	83.0%
